Simulamos nulos y hacemos interpolación lineal

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado. Asegúrate de que esté en la carpeta 'CSVs' dentro de tu proyecto.")
    df_air_quality = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv en la ruta esperada.")
    print("Creando DataFrame de ejemplo para continuar con la ejecución del script...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200): # Reducido para ejemplo rápido
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([
                _date, _city_ex,
                200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5,
                50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20,
                40+np.random.rand()*20
            ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
if 'CO2' in df_air_quality.columns:
    df_air_quality = df_air_quality.drop(columns=['CO2'])
    print("Columna 'CO2' descartada.")

# --- Parámetros de Simulación e Imputación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL
N_REPETITIONS = 5

NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
METRICS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'metricas')
IMPUTED_CSVS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'CSVs')

os.makedirs(SIMULATED_DATA_ROOT_DIR, exist_ok=True)
os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares ---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    np.random.seed(random_seed_input)
    random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True)
    n_rows = len(df_with_nans)
    row_indices_to_nan_positions = []
    scenario_type = scenario_config_input['type']
    config = scenario_config_input['config']
    if scenario_type == 'row_percentage':
        percentage = config['percentage']
        num_rows_to_nan = int(n_rows * percentage)
        if num_rows_to_nan > 0 and n_rows > 0 :
            row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length = config['length']
        num_blocks_ratio = config.get('num_blocks_ratio', 0.01)
        num_blocks = max(1, int( (n_rows * num_blocks_ratio) / block_length if block_length > 0 else n_rows * num_blocks_ratio ))
        for _ in range(num_blocks):
            if n_rows < block_length: break
            if (n_rows - block_length) < 0: continue
            start_pos = random.randint(0, n_rows - block_length)
            current_block_positions = list(range(start_pos, start_pos + block_length))
            row_indices_to_nan_positions.extend(current_block_positions)
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))
    if row_indices_to_nan_positions:
        col_positions_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_to_nan = [pos for pos in col_positions_to_nan if pos != -1]
        if col_positions_to_nan:
             df_with_nans.iloc[row_indices_to_nan_positions, col_positions_to_nan] = np.nan
    df_original_city_reset = df_original_city_input.copy().reset_index(drop=True)
    cols_to_check_mask = [col for col in ALL_TARGET_COLS if col in df_with_nans.columns and col in df_original_city_reset.columns]
    if not cols_to_check_mask:
        introduced_nan_mask_df = pd.DataFrame(index=df_with_nans.index)
    else:
        introduced_nan_mask_df = df_with_nans[cols_to_check_mask].isnull() & ~df_original_city_reset[cols_to_check_mask].isnull()
    return df_with_nans, introduced_nan_mask_df

def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {} # Usar un diccionario para almacenar todas las métricas por columna
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0:
            continue
        
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0:
            continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            
            # Calcular rango de la variable original (sobre los valores verdaderos donde se imputó)
            # O mejor, sobre toda la columna original para tener un denominador consistente
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            
            nrmae_range = np.nan
            nrmse_range = np.nan
            if original_col_range > 1e-9: # Evitar división por cero o rango muy pequeño
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range

            results[col]['MAE'] = mae
            results[col]['RMSE'] = rmse
            results[col]['NRMAE_range'] = nrmae_range
            results[col]['NRMSE_range'] = nrmse_range

        except Exception as e:
            # print(f"Error calculando métricas para {col}: {e}") # Comentado
            pass # Los NaNs se mantendrán
            
    return results

# --- Script Principal ---
all_detailed_metrics_results = []
start_time_script = time.time()

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está cargado o está vacío. Carga tus datos primero.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original} (Directorio seguro: {city_name_safe})")
    city_sim_data_dir_root = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    os.makedirs(city_sim_data_dir_root, exist_ok=True)
    city_imputed_csv_dir_root = os.path.join(IMPUTED_CSVS_OUTPUT_DIR, city_name_safe)
    os.makedirs(city_imputed_csv_dir_root, exist_ok=True)

    df_city_original_for_sim = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city_original_for_sim.empty:
        print(f"  No hay datos para la ciudad {city_name_original}. Omitiendo.")
        continue
    if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Advertencia: NaNs preexistentes en {city_name_original}. Rellenando con ffill/bfill para ground truth de simulación.")
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_original_for_sim.columns:
                df_city_original_for_sim[col_fill] = df_city_original_for_sim[col_fill].ffill().bfill()
        if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
            print(f"  Error: No se pudieron rellenar NaNs preexistentes en {city_name_original} tras ffill/bfill. Omitiendo.")
            continue
    df_city_original_ground_truth_reset = df_city_original_for_sim.copy().reset_index(drop=True)

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        # print(f"  Escenario de Simulación de NaNs: {scenario_name_safe}") # Comentado
        scenario_city_sim_data_dir_specific = os.path.join(city_sim_data_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_sim_data_dir_specific, exist_ok=True)
        scenario_city_imputed_csv_dir_specific = os.path.join(city_imputed_csv_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_imputed_csv_dir_specific, exist_ok=True)

        for rep in range(1, N_REPETITIONS + 1):
            current_seed = int(time.time_ns() + random.randint(0,100000) + rep + hash(city_name_original) + hash(scenario_name_safe)) % (2**32 - 1)
            df_city_with_nans_reset, introduced_nan_mask_df_reset = introduce_nans_by_row(
                df_city_original_for_sim, scenario, random_seed_input=current_seed
            )
            sim_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_with_nans.csv"
            df_city_with_nans_reset.to_csv(os.path.join(scenario_city_sim_data_dir_specific, sim_filename), index=False)
            
            df_city_imputed_linear_reset = df_city_with_nans_reset.copy()
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].interpolate(
                        method='linear', limit_direction='both', limit_area=None)
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].ffill().bfill()
            imputed_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_imputed_linear.csv"
            df_city_imputed_linear_reset.to_csv(os.path.join(scenario_city_imputed_csv_dir_specific, imputed_filename), index=False)

            metrics_scores = calculate_metrics_with_nmae_nrmse( # Nueva función de métricas
                df_city_original_ground_truth_reset,
                df_city_imputed_linear_reset,
                introduced_nan_mask_df_reset
            )
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed, 'Imputation_Method': INTERPOLATION_METHOD_NAME,
                    'Variable': var_name, 
                    'MAE': metrics_scores[var_name]['MAE'], 
                    'RMSE': metrics_scores[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores[var_name]['NRMAE_range'], # Nueva métrica
                    'NRMSE_range': metrics_scores[var_name]['NRMSE_range']  # Nueva métrica
                })
    print(f"  Procesamiento de escenarios para {city_name_original} completado.")


# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_results)
    
    # 1. metrics_variables_summary_<ciudad>.csv
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    # Ahora incluirá NRMAE_range y NRMSE_range en el promedio
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={
        'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 
        'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    
    for city_name_original in grouped_by_city_var['City'].unique():
        city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_original]
        city_metrics_df_to_save = city_metrics_df[['City', 'Variable', 'Avg_MAE', 'Avg_RMSE', 'Avg_NRMAE_range', 'Avg_NRMSE_range']]
        variables_summary_path = os.path.join(METRICS_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df_to_save.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_original} guardado en: {variables_summary_path}")
        if city_name_original == df_air_quality['City'].unique()[0]: # Imprimir solo para la primera ciudad como ejemplo
            print(city_metrics_df_to_save)


    # 2. metrics_contaminants_combined_summary.csv
    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(CONTAMINANT_COLS)]
    # Promediar todas las métricas, incluyendo las normalizadas
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={
        'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 
        'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
    print(f"  Resumen combinado de contaminantes guardado en: {contaminants_summary_path}")
    print(contaminants_combined_summary)

    # 3. metrics_aqi_combined_summary.csv
    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    # Promediar todas las métricas, incluyendo las normalizadas
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={
        'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI',
        'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False)
    print(f"  Resumen combinado de AQI guardado en: {aqi_summary_path}")
    print(aqi_combined_summary)
    
    # (Opcional) Guardar el ALL_metrics_detailed si aún lo quieres para inspección manual
    # global_detailed_metrics_file = os.path.join(METRICS_OUTPUT_DIR, f'ALL_metrics_detailed_{INTERPOLATION_METHOD_NAME}.csv')
    # df_all_detailed_metrics.to_csv(global_detailed_metrics_file, index=False)
    # print(f"\nTodas las métricas detalladas guardadas en: {global_detailed_metrics_file}")
else:
    print("No se generaron resultados de métricas detalladas.")

end_time_script = time.time()
print(f"\nProceso completado.")
print(f"Archivos de datos simulados guardados en subcarpetas dentro de: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos de métricas resumidas guardados en: {METRICS_OUTPUT_DIR}")
print(f"Archivos CSV imputados guardados en subcarpetas dentro de: {IMPUTED_CSVS_OUTPUT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv
Columna 'CO2' descartada.

Procesando Ciudad: Brasilia (Directorio seguro: Brasilia)
  Procesamiento de escenarios para Brasilia completado.

Procesando Ciudad: Cairo (Directorio seguro: Cairo)
  Procesamiento de escenarios para Cairo completado.

Procesando Ciudad: Dubai (Directorio seguro: Dubai)
  Procesamiento de escenarios para Dubai completado.

Procesando Ciudad: London (Directorio seguro: London)
  Procesamiento de escenarios para London completado.

Procesando Ciudad: New York (Directorio seguro: New_York)
  Procesamiento de escenarios para New York completado.

Procesando Ciudad: Sydney (Directorio seguro: Sydney)
  Procesamiento de escenarios para Sydney completado.

Generando resúmenes de métricas por variable y ciudad...
  Resumen de variables para Brasilia guardado en: c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\imputaciones\interpolacionLineal\metricas\metrics_variables_summary_Brasilia.csv
  Resumen de variables par